# DP_task_RAG · Advanced + Agentic RAG over Harvey LAB (antitrust + deal-regulatory) on a free Colab T4

**Runtime → Change runtime type → T4 GPU**, then run top to bottom (~15 min plus the eval you choose).

| Section | What happens | GPU / LLM |
|---|---|---|
| 1. Setup | clone, install, load Claude credentials from Colab Secrets | – |
| 2. Indexes | BGE-M3 dense + BM25 for 3 text variants: `raw` / `header` / `ctx` (Contextual Retrieval) | T4, ~1.5 min per variant |
| 3. Observability | Arize Phoenix in the notebook, LangGraph auto-instrumented + manual retriever/LLM spans | – |
| 4. Demo | the agent on a *simple*, a *complex* and a *direct* question, showing its decision log | T4 + Claude |
| 5. Evaluation | retrieval ablations, CRAG calibration (dev), generation eval on the held-out **test** split (single-shot vs. agent), Adaptive-RAG ablation, unanswerable questions | T4 + Claude |

**Credentials** (🔑 Secrets, enable notebook access): `CLAUDE_CODE_OAUTH_TOKEN` (from `claude setup-token`, uses your
Claude subscription through `claude_agent_sdk`) **or** `ANTHROPIC_API_KEY` (fallback, Anthropic API). Nothing is printed.

The one-off Contextual Retrieval job (Qwen3.5-4B on vLLM, ~2–2.5 h) lives in `01_contextualize_colab.ipynb`. Its
output is committed as `artifacts/contexts.jsonl` or picked up from Drive, so this notebook never re-runs it. The agent and
the eval run on `header`: the Qwen3.5-4B contexts didn't match it on dev retrieval (docs/03), so `ctx` is built
only for the retrieval ablation.

**Corpus:** all 31 antitrust matters plus 13 deal-regulatory matters (merger control / HSR / SPA from corporate-ma,
CFIUS from international-trade-sanctions): 318 documents, 8,627 chunks. **Eval splits:** `dev` = the 110 questions that
shaped the design (prompt fixes, CRAG thresholds); `test` = questions for 16 matters never used while building it.

## 1. Setup

In [ ]:
import os, subprocess, sys, json
REPO = "https://github.com/nrusnac/DP_task_RAG.git"
BRANCH = "agent-improvements"
if not os.path.exists("/content/DP_task_RAG"):
    subprocess.run(["git", "clone", "-q", REPO, "/content/DP_task_RAG"], check=True)
os.chdir("/content/DP_task_RAG")
# sync to the latest commit of BRANCH; outputs of earlier runs in this clone are discarded (they are saved separately)
subprocess.run(["git", "fetch", "-q", "origin", BRANCH], check=True)
subprocess.run(["git", "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "reset", "-q", "--hard", f"origin/{BRANCH}"], check=True)
!git log --oneline -1
!pip -q install -e ".[obs]"
# an editable install is only picked up by NEW processes; make it importable in this running kernel
sys.path.insert(0, "/content/DP_task_RAG/src")


In [ ]:
import hrag.config  # sets JAX_PLATFORMS=cpu before bm25s is imported (Colab's GPU JAX would grab 75% of VRAM)
from hrag import llm
print("secrets found:", llm.load_colab_secrets(), "| backend:", llm.backend())
print(open("artifacts/ingest_stats.json").read())

## 2. Indexes
Text variants for the Contextual Retrieval ablation: `raw` (chunk only), `header` (deterministic
`[doc title] (type) › section` + chunk), and `ctx` (header + Qwen-written context + chunk). If `contexts.jsonl` isn't
in the repo yet, it's looked up on Drive (where notebook 01 writes `contexts_<model>.jsonl`).

In [ ]:
from pathlib import Path
import glob, shutil
from hrag.config import CFG
ctx = Path("artifacts/contexts.jsonl")
if not ctx.exists():
    found = sorted(glob.glob("/content/drive/MyDrive/dp_task_rag/contexts_*.jsonl"), key=os.path.getmtime)
    want = f"/content/drive/MyDrive/dp_task_rag/contexts_{CFG.context_model.split('/')[-1].lower()}.jsonl"
    src = want if want in found else (found[-1] if found else None)  # the configured model, else the newest file
    if src:
        shutil.copy(src, ctx); print("contexts from Drive:", src)
variants = ["raw", "header"] + (["ctx"] if ctx.exists() else [])
print("building:", variants)
!python -m hrag.index.build --variants {" ".join(variants)}

## 3. Observability (Phoenix)

In [ ]:
from hrag.obs import tracing
print("Phoenix UI →", tracing.setup(project="dp-task-rag"))

## 4. Demo: the agentic module
Watch the `log`: the Adaptive-RAG route, the CRAG verdict per sub-question (with the best rerank score), corrective
rewrites, and sufficiency iterations. Every step is also a span in Phoenix.

In [ ]:
from hrag.agent.graph import answer
from IPython.display import Markdown, display
# the agent and the eval use the header index: the Qwen3.5-4B contexts did not match it on dev retrieval
# (docs/03, "Contextual Retrieval"); ctx is still built above for the retrieval ablation in 5a
VARIANT = "header"

async def show(q, task_id, **kw):
    r = await answer(q, task_id=task_id, variant=VARIANT, **kw)
    display(Markdown(f"**Q:** {q}\n\n**route:** `{r['route']}` · **LLM calls:** {r['llm_calls']} · **{r['seconds']} s**\n\n"
                     + "\n".join(f"- {l}" for l in r["log"]) + "\n\n---\n" + (r["answer"] or "")
                     + "\n\n**Sources:** " + ", ".join(f"[{c['n']}] {c['doc']}" for c in r["citations"])))
    return r

T = "analyze-iss-antitrust-transaction-structure"
r1 = await show("What are the post-merger HHI and the HHI delta for flame retardants in the South Central region?", T)

In [ ]:
r2 = await show("Which product markets raise presumptive concerns under the 2023 Merger Guidelines, and what remedies "
                "and HSR filing strategy should the client consider?", T)

In [ ]:
r3 = await show("What can you help me with?", T)

**CRAG's corrective path:** a question the data room can't answer should be graded *incorrect*, rewritten (keywords + HyDE), retried, and then **abstained on** instead of answered from look-alike evidence.

In [ ]:
r4 = await show("What annual membership fee does the target pay for its CEO's country club, and who approved it?", T)

## 5. Evaluation
The eval set (`artifacts/eval/qa.jsonl`) was built from Harvey rubric criteria: Claude rewrote each criterion as a
question plus verbatim key facts, and chunks containing those facts are the **silver gold labels** (when the facts
match more than 8 chunks, Claude picks the ones that answer the question) (`python -m hrag.eval.build_qa`).
Two splits: **dev** (110 questions; used to tune prompts and calibrate CRAG) and **test** (matters never used while
building the system: the honest numbers). See `docs/05-evaluation.md`.

**Claude usage:** 5c ≈ 1,050 calls (naive RAG is no longer run here: its numbers on main and on the first
test run are in `docs/05-evaluation.md`), 5d ≈ 270, 5e ≈ 280. A subscription session limit stops a cell cleanly
(`STOPPED: ... session limit`); after the limit resets, **re-run the same cell**: `resume=True` keeps every finished
answer and runs only the missing ones. Results are also on Drive if you copy `artifacts/eval` there (last cell).

In [ ]:
from hrag.eval import run_eval
import pandas as pd
qa = run_eval.load_qa()
for split in run_eval.SPLITS:
    s = [i for i in qa if i.get("split", "dev") == split]
    print(f"{split}: {len(s)} QA items, {sum(bool(i['gold']) for i in s)} with silver retrieval labels, "
          f"{len({i['task_id'] for i in s})} matters")
ret = run_eval.retrieval(variants)          # 5a. retrieval ablations per split: no LLM calls

In [ ]:
def table(d, cols=None):
    """Summary dict -> DataFrame; tolerates an empty or partial run."""
    df = pd.DataFrame(d).T
    return df[[c for c in cols if c in df.columns]] if cols and not df.empty else df

table(ret, ["recall@1", "recall@5", "recall@10", "mrr", "ndcg@10", "foreign@5", "ms_per_query"])

In [ ]:
cal = run_eval.calibrate(VARIANT)           # 5b. CRAG thresholds on the dev split (same-practice-area negatives)

In [ ]:
# 5c. generation eval on the held-out test split: advanced single-shot RAG vs. the agent
# (Harvey judge + faithfulness). N_TEST = None runs every test question; set e.g. 40 for a quicker pass.
# If it stops at a session limit, re-run this cell after the reset: finished answers are kept.
# tag "test_v2": the run after the over-hedging / evidence fixes ("test" was the first run, before them)
N_TEST = None
gen = await run_eval.generation(n=N_TEST, systems=["single_shot", "agent"], variant=VARIANT,
                                split="test", tag="test_v2", resume=True)
table(gen)

In [ ]:
# The first test run (before the fixes, artifacts/eval/gen_test_v1.jsonl) was read question by question to find what
# went wrong, so its 83 questions are no longer fully held out. The test questions it never reached still are:
seen = {json.loads(l)["qid"] for l in open("artifacts/eval/gen_test_v1.jsonl")}
v2 = [json.loads(l) for l in open("artifacts/eval/gen_test_v2.jsonl")]
print(len({r["qid"] for r in v2} - seen), "test questions never seen during the diagnosis")
table(run_eval.summarize_generation([r for r in v2 if r["qid"] not in seen]))

In [ ]:
# 5d. Adaptive-RAG ablation (test split): adaptive routing vs. forcing every query down one path
abl = await run_eval.generation(n=20, systems=["agent:simple", "agent:complex", "agent"], variant=VARIANT,
                                split="test", tag="ablation_test_v2", judge_faith=False, resume=True)
table(abl, ["pass_rate", "avg_llm_calls", "avg_seconds", "routes"])

5e. Unanswerable questions (test split): each question asked in ANOTHER matter's data room of the same practice area. A correct system declines instead of answering from look-alike evidence.

In [ ]:
neg = await run_eval.negatives(n=40, systems=["single_shot", "agent"], variant=VARIANT, split="test", tag="test_v2",
                              resume=True)
table(neg)

In [ ]:
print("LLM usage this session:", llm.STATS.snapshot())
g = globals().get
print(json.dumps({"retrieval": g("ret"), "calibration": (g("cal") or {}).get("suggested"), "generation": g("gen"),
                  "adaptive_ablation": g("abl"), "unanswerable": g("neg")}, indent=1, default=str)[:6000])

In [ ]:
# Save the results: download the zip and unzip it into the repo's artifacts/eval/ (also copied to Drive if mounted)
import shutil, os
shutil.make_archive("/content/eval_results", "zip", "/content/DP_task_RAG/artifacts", "eval")
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs("/content/drive/MyDrive/dp_task_rag", exist_ok=True)
    shutil.copy("/content/eval_results.zip", "/content/drive/MyDrive/dp_task_rag/eval_results.zip")
from google.colab import files
files.download("/content/eval_results.zip")